In [1]:
import pandas as pd
import os

PROCESSED_INPUT = "C:/Users/julie/Data Eng/Raw/Processed/events_all.csv"
OUTPUT_FILE = "C:/Users/julie/Data Eng/Processed/deliveries.csv"
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

def transform_events_to_deliveries():
    df = pd.read_csv(PROCESSED_INPUT, parse_dates=["timestamp"])
    df = df.sort_values(["delivery_id", "timestamp"])
    
    deliveries = []

    for delivery_id, group in df.groupby("delivery_id"):
        start_time = group["timestamp"].min()
        end_time = group["timestamp"].max()
        client_id = group["client_id"].iloc[0]
        driver_id = group["driver_id"].iloc[0]

        if "delivery_success" in group["event"].values:
            status = "success"
            duration = group[group["event"] == "delivery_success"]["duration_minutes"].values[0]
            failure_reason = None
        elif "delivery_failed" in group["event"].values:
            status = "failed"
            duration = group[group["event"] == "delivery_failed"]["duration_minutes"].values[0]
            failure_reason = group[group["event"] == "delivery_failed"]["failure_reason"].values[0]
        else:
            status = "in_progress"
            duration = None
            failure_reason = None

        deliveries.append({
            "delivery_id": delivery_id,
            "client_id": client_id,
            "driver_id": driver_id,
            "start_time": start_time,
            "end_time": end_time,
            "status": status,
            "duration_minutes": duration,
            "failure_reason": failure_reason,
            "event_count": len(group)
        })

    df_deliveries = pd.DataFrame(deliveries)
    df_deliveries.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ {len(df_deliveries)} livraisons consolidées dans {OUTPUT_FILE}")

if __name__ == "__main__":
    transform_events_to_deliveries()


✅ 100 livraisons consolidées dans C:/Users/julie/Data Eng/Processed/deliveries.csv
